# Semana 10 — Lista de Exercícios de SQL Avançado para Data Warehouse

Esta lista foi revisada considerando os conteúdos práticos das Aulas 1 e 2 e a Aula 3 como conteúdo teórico.

## Modelo de dados usado

- `fato_vendas`
- `dim_cliente`
- `dim_produto`
- `dim_tempo`

## Conteúdos praticados

- Subconsultas
- CTEs com `WITH`
- Window Functions
- Índices e otimização
- Conceitos teóricos de particionamento e materialized views

## 0. Preparação do ambiente

In [1]:
import duckdb
import pandas as pd

# Ajuste o caminho dos arquivos se necessário.
# Se os CSVs estiverem na mesma pasta do notebook, este código já deve funcionar.

dim_cliente = pd.read_csv('../Dataset_schema/dim_cliente.csv')
dim_produto = pd.read_csv('../Dataset_schema/dim_produto.csv')
dim_tempo = pd.read_csv('../Dataset_schema/dim_tempo.csv')
fato_vendas = pd.read_csv('../Dataset_schema/fato_vendas.csv')

banco_dados = duckdb.connect()

# Registra os DataFrames como tabelas consultáveis pelo DuckDB
banco_dados.register('dim_cliente', dim_cliente)
banco_dados.register('dim_produto', dim_produto)
banco_dados.register('dim_tempo', dim_tempo)
banco_dados.register('fato_vendas', fato_vendas)

# Para a parte de índices, precisamos de tabelas físicas no DuckDB.
# Por isso criamos versões com o sufixo _base.
banco_dados.sql("DROP TABLE IF EXISTS fato_vendas_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_cliente_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_produto_base")
banco_dados.sql("DROP TABLE IF EXISTS dim_tempo_base")

banco_dados.sql("CREATE TABLE fato_vendas_base AS SELECT * FROM fato_vendas")
banco_dados.sql("CREATE TABLE dim_cliente_base AS SELECT * FROM dim_cliente")
banco_dados.sql("CREATE TABLE dim_produto_base AS SELECT * FROM dim_produto")
banco_dados.sql("CREATE TABLE dim_tempo_base AS SELECT * FROM dim_tempo")

---

# Parte 1 — Subconsultas

## Exercício 1 — Vendas acima da média geral

**Nível:** Fácil

Liste as vendas cujo `valor_total` seja maior que a média geral de `valor_total`.

Retorne:
- `venda_sk`
- `order_id`
- `cliente_sk`
- `valor_total`

Ordene por `valor_total` em ordem decrescente.

**Dica:** a subconsulta deve retornar apenas um valor: a média geral de `valor_total`.

In [14]:
# Escreva sua consulta SQL aqui

query = """
SELECT
	venda_sk,
    order_id,
    cliente_sk,
    valor_total
FROM fato_vendas
WHERE valor_total > (
	SELECT
		AVG (valor_total)
    FROM fato_vendas
)
ORDER BY valor_total DESC
"""

# Para executar:
banco_dados.sql(query).df()

,venda_sk,order_id,cliente_sk,valor_total
0,3482,0812eb902a67711a1cb742b3cdaa65ae,13463,6929.31
1,109785,fefacc66af859508bf1a7934eab1e97f,21172,6922.21
2,105496,f5136e38d1a14a4dbd87dff67da82701,3907,6726.66
3,72721,a96610ab360d42a2e5335a3998b4718a,27305,4950.34
4,10998,199af31afc78c699f0dbf71fb178d4d4,12062,4764.34
...,...,...,...,...
32858,62148,911c0c489bf4354f38fc64491075e8ba,19602,139.93
32859,62149,911c0c489bf4354f38fc64491075e8ba,19602,139.93
32860,93983,d9f134a34fb4b3cfd95550b4785e8d05,1659,139.93
32861,81520,bd3b1eb371dd6c8ca77447b363de08cd,83194,139.93


## Exercício 2 — Vendas com frete acima da média geral

**Nível:** Fácil/Médio

Liste as vendas cujo `valor_frete` seja maior que a média geral de frete.

Retorne:
- `venda_sk`
- `order_id`
- `valor_frete`
- `valor_total`

Ordene por `valor_frete` em ordem decrescente.

**Dica:** use uma subconsulta dentro do `WHERE`.

In [17]:
# Escreva sua consulta SQL aqui

query = """
SELECT
	venda_sk,
    order_id,
    valor_frete,
    valor_total
FROM fato_vendas
WHERE valor_frete > (
	SELECT
		AVG(valor_frete)
    FROM fato_vendas
)
ORDER BY valor_frete DESC
"""

# Para executar:
banco_dados.sql(query).df().round(2)

,venda_sk,order_id,valor_frete,valor_total
0,71885,a77e1550db865202c56b19ddc6dc4d53,409.68,1388.68
1,3232,076d1555fb53a89b0ef4d529e527a0f6,375.28,2713.36
2,27413,3fde74c28a3d5d618c00f26d51baafa0,375.28,2713.36
3,68275,9f49bd16053df810384e793386312674,339.59,1488.59
4,16360,264a7e199467906c0727394df82d1a6a,338.30,1388.30
...,...,...,...,...
30676,88976,ce8d55aa1c206156fde37e062c716ea2,19.95,35.95
30677,66169,9abbdc48fa5090fc71433b094fd329a8,19.95,84.85
30678,97683,e26cd342669df630e929534f52f9751a,19.95,159.94
30679,85286,c5e7393e8d0c01dc92c41bc4684f1c45,19.95,39.94


## Exercício 3 — Categorias com quantidade de vendas acima da média

**Nível:** Médio

Calcule a quantidade de vendas por categoria e retorne apenas as categorias com quantidade de vendas acima da média das categorias.

Retorne:
- `categoria`
- `qtd_vendas`

**Dica:** primeiro pense no agrupamento por categoria. Depois, use uma subconsulta para comparar com a média dessas quantidades.

In [24]:
# Escreva sua consulta SQL aqui

query = """
SELECT
	categoria,
    qtd_vendas
FROM (
	SELECT
		p.product_category_name AS categoria,
        COUNT(*) AS qtd_vendas
    FROM fato_vendas f
    JOIN dim_produto p
		ON f.produto_sk = p.produto_sk
    WHERE p.product_category_name IS NOT NULL
    GROUP BY categoria
) categorias
WHERE qtd_vendas > (
	SELECT
		AVG(qtd_vendas)
    FROM (
    	SELECT
			COUNT(*) AS qtd_vendas
        FROM fato_vendas f
        JOIN dim_produto p
			ON f.produto_sk = p.produto_sk
        WHERE product_category_name IS NOT NULL
        GROUP BY p.product_category_name
    ) media_categorias
)
ORDER BY qtd_vendas DESC
"""

# Para executar:
banco_dados.sql(query).df()

,categoria,qtd_vendas
0,cama_mesa_banho,10953
1,beleza_saude,9465
2,esporte_lazer,8431
3,moveis_decoracao,8160
4,informatica_acessorios,7644
5,utilidades_domesticas,6795
6,relogios_presentes,5859
7,telefonia,4430
8,ferramentas_jardim,4268
9,automotivo,4140


## Exercício 4 — Vendas dos 3 produtos mais vendidos

**Nível:** Médio/Difícil

Retorne as vendas dos 3 produtos que mais aparecem na `fato_vendas`.

Retorne:
- `venda_sk`
- `order_id`
- `produto_sk`
- `valor_total`

**Dica:** a subconsulta deve descobrir quais são os 3 `produto_sk` com maior quantidade de vendas.

In [38]:
# Escreva sua consulta SQL aqui

query = """
SELECT
	venda_sk,
    order_id,
    produto_sk,
    valor_total
FROM fato_vendas
WHERE produto_sk IN (
	SELECT
		produto_sk
    FROM fato_vendas
    GROUP BY produto_sk
    ORDER BY COUNT(*) DESC
    LIMIT 3
)
ORDER BY produto_sk, valor_total DESC
"""

# Para executar:
banco_dados.sql(query).df().round(2)

,venda_sk,order_id,produto_sk,valor_total
0,26174,3ce7418ccb9af136b4937f069601ab2f,9662,137.23
1,85118,c5883e5be24c4f0627d36a56ed34d190,9662,132.94
2,87947,cc2e6cf82767599713d657cd5cf8b300,9662,132.94
3,6271,0e8594074c3143aa6b56a3ec70a46e12,9662,132.94
4,26723,3e2b45761c1280a1c1165c83d3292c43,9662,132.94
...,...,...,...,...
1476,52995,7b214cd39f1ef2aaeb78dbbffc63439d,14052,53.90
1477,57671,86509c09dbafbcba748558e5f36d1e7c,14052,53.90
1478,18661,2bd6a9cea9df97f5f2d0717bb6307f32,14052,53.90
1479,505,012b3f6ab7776a8ab3443a4ad7bef2e6,14052,53.90


## Exercício 5 — Estados com receita acima da média dos estados

**Nível:** Difícil

Calcule a receita total por estado e liste apenas os estados cuja receita total esteja acima da média de receita entre os estados.

Retorne:
- `estado`
- `receita_total`

**Dica:** você pode usar uma subconsulta com outra consulta agregada por estado.

In [40]:
# Escreva sua consulta SQL aqui

query = """
SELECT
	estado,
    receita_total
FROM (
	SELECT
		c.customer_state AS estado,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    JOIN dim_cliente c
		ON f.cliente_sk = c.cliente_sk
    GROUP BY estado
) receita_estados
WHERE receita_total > (
	SELECT
		AVG(receita_total)
    FROM (
		SELECT
			SUM(f.valor_total) AS receita_total
        FROM fato_vendas f
        JOIN dim_cliente c
			ON f.cliente_sk = c.cliente_sk
        GROUP BY c.customer_state
    ) media_estados
)
ORDER BY receita_total DESC
"""

# Para executar:
banco_dados.sql(query).df().round(2)

,estado,receita_total
0,SP,5769703.15
1,RJ,2055401.57
2,MG,1818891.67
3,RS,861472.79
4,PR,781708.80
5,SC,595127.78
6,BA,591137.81


---

# Parte 2 — CTEs com WITH

## Exercício 6 — Vendas por estado usando CTE

**Nível:** Fácil

Crie uma CTE chamada `vendas_por_estado` com:
- quantidade de pedidos
- quantidade de itens vendidos
- receita total por estado

Depois, consulte a CTE ordenando por maior receita.

**Dica:** a CTE deve fazer o `JOIN` entre `fato_vendas` e `dim_cliente`.

In [48]:
# Escreva sua consulta SQL aqui

query = """
WITH vendas_por_estado AS (
	SELECT
		c.customer_state AS estado,
		COUNT(DISTINCT f.order_id) AS qtd_pedidos,
        COUNT(order_item_id) AS qtd_itens_vendidos,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    JOIN dim_cliente c
		ON f.cliente_sk = c.cliente_sk
    GROUP BY estado
)

SELECT *
	FROM vendas_por_estado
ORDER BY receita_total DESC
"""

# Para executar:
banco_dados.sql(query).df().round(2)

,estado,qtd_pedidos,qtd_itens_vendidos,receita_total
0,SP,40501,46448,5769703.15
1,RJ,12350,14143,2055401.57
2,MG,11354,12916,1818891.67
3,RS,5345,6134,861472.79
4,PR,4923,5649,781708.80
5,SC,3546,4097,595127.78
6,BA,3256,3683,591137.81
7,DF,2080,2355,346123.35
8,GO,1957,2277,334212.35
9,ES,1995,2225,317657.93


## Exercício 7 — Categorias com maior faturamento usando CTE

**Nível:** Médio

Crie uma CTE chamada `resumo_categorias` para calcular:
- quantidade de vendas
- receita total
- ticket médio por categoria

Depois, na consulta final, mostre apenas categorias com receita total acima de R$ 100.000.

**Dica:** a CTE calcula o resumo; a consulta final aplica o filtro.

In [54]:
# Escreva sua consulta SQL aqui

query = """
WITH resumo_categorias AS (
	SELECT
		p.product_category_name AS categoria,
        COUNT(DISTINCT f.order_id) AS qtd_vendas,
        SUM(f.valor_total) AS receita_total,
        AVG(f.valor_total) AS ticket_medio
    FROM fato_vendas f
    JOIN dim_produto p
		ON f.produto_sk = p.produto_sk
    GROUP BY categoria
)

SELECT *
FROM resumo_categorias
WHERE receita_total > 100000
ORDER BY receita_total DESC
"""

# Para executar:
banco_dados.sql(query).df().round(2)

,categoria,qtd_vendas,receita_total,ticket_medio
0,beleza_saude,8647,1412089.53,149.19
1,relogios_presentes,5495,1264333.12,215.79
2,cama_mesa_banho,9272,1225209.26,111.86
3,esporte_lazer,7530,1118256.91,132.64
4,informatica_acessorios,6530,1032723.77,135.10
5,moveis_decoracao,6307,880329.92,107.88
6,utilidades_domesticas,5743,758392.25,111.61
7,cool_stuff,3559,691680.89,186.04
8,automotivo,3810,669454.75,161.70
9,ferramentas_jardim,3448,567145.68,132.88


## Exercício 8 — Receita por estado e categoria usando CTE

**Nível:** Difícil

Crie uma CTE para calcular a receita total e a quantidade de vendas por estado e categoria de produto.

Depois, retorne apenas as combinações com receita acima de R$ 50.000.

Retorne:
- `estado`
- `categoria`
- `quantidade_vendas`
- `receita_total`

**Dica:** use `fato_vendas`, `dim_cliente` e `dim_produto`.

In [60]:
# Escreva sua consulta SQL aqui

query = """
WITH receita_total_estado_categoria AS (
	SELECT
		c.customer_state AS estado,
        p.product_category_name AS categoria,
        COUNT(f.order_id) AS qtd_vendas,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    JOIN dim_cliente c
		ON f.cliente_sk = c.cliente_sk
    JOIN dim_produto p
		ON f.produto_sk = p.produto_sk
    GROUP BY estado, categoria
)

SELECT *
FROM receita_total_estado_categoria
WHERE receita_total > 50000
ORDER BY estado, receita_total DESC
"""

# Para executar:
banco_dados.sql(query).df().round(2)

,estado,categoria,qtd_vendas,receita_total
0,BA,beleza_saude,340,58620.07
1,BA,relogios_presentes,225,50254.70
2,MG,beleza_saude,1064,175007.52
3,MG,cama_mesa_banho,1322,156483.10
4,MG,relogios_presentes,626,132311.65
...,...,...,...,...
61,SP,construcao_ferramentas_construcao,422,66397.74
62,SP,consoles_games,472,65789.05
63,SP,fashion_bolsas_e_acessorios,841,65487.98
64,SP,pcs,59,64371.71


## Exercício 9 — Meses com receita acima da média mensal

**Nível:** Médio/Difícil

Crie uma CTE chamada `receita_mensal` para calcular a receita total por mês.

Depois, retorne apenas os meses cuja receita total esteja acima da média mensal.

Retorne:
- `ano_mes`
- `receita_total`

**Dica:** a média deve ser calculada a partir da própria CTE.

In [66]:
# Escreva sua consulta SQL aqui

query = """
WITH receita_mensal AS (
	SELECT
		t.ano_mes,
        SUM(f.valor_total) AS receita_total
    FROM fato_vendas f
    JOIN dim_tempo t
		ON f.tempo_sk = t.tempo_sk
    GROUP BY t.ano_mes
),

media_mensal AS (
	SELECT
    	AVG(receita_total) AS media_receita_mensal
    FROM receita_mensal
)

SELECT
	r.ano_mes,
    r.receita_total
FROM receita_mensal r
CROSS JOIN media_mensal m
WHERE r.receita_total > m.media_receita_mensal
ORDER BY r.ano_mes
"""

# Para executar:
banco_dados.sql(query).df().round(2)

,ano_mes,receita_total
0,2017-09,701077.49
1,2017-10,751117.01
2,2017-11,1153364.20
3,2017-12,843078.29
4,2018-01,1077887.46
5,2018-02,966168.41
6,2018-03,1120598.24
7,2018-04,1132878.93
8,2018-05,1128774.52
9,2018-06,1011978.29


## Exercício 10 — Pipeline de análise por categoria

**Nível:** Difícil

Crie CTEs para responder:

Quais categorias têm receita total acima de R$ 50.000 e ticket médio acima de R$ 100?

Retorne:
- `categoria`
- `qtd_vendas`
- `receita_total`
- `ticket_medio`

**Dica:** use uma CTE para calcular as métricas por categoria e a consulta final para filtrar.

In [74]:
# Escreva sua consulta SQL aqui

query = """
WITH metricas_categorias AS (
	SELECT
		p.product_category_name AS categoria,
        COUNT(DISTINCT f.order_id) AS qtd_vendas,
        SUM(f.valor_total) AS receita_total,
        AVG(f.valor_total) AS ticket_medio
	FROM fato_vendas f
    JOIN dim_produto p
		ON f.produto_sk = p.produto_sk
    WHERE categoria IS NOT NULL
    GROUP BY categoria
)
SELECT *
FROM metricas_categorias
WHERE receita_total > 50000 AND ticket_medio > 100
ORDER BY receita_total DESC
"""

# Para executar:
banco_dados.sql(query).df().round(2)

,categoria,qtd_vendas,receita_total,ticket_medio
0,beleza_saude,8647,1412089.53,149.19
1,relogios_presentes,5495,1264333.12,215.79
2,cama_mesa_banho,9272,1225209.26,111.86
3,esporte_lazer,7530,1118256.91,132.64
4,informatica_acessorios,6530,1032723.77,135.10
5,moveis_decoracao,6307,880329.92,107.88
6,utilidades_domesticas,5743,758392.25,111.61
7,cool_stuff,3559,691680.89,186.04
8,automotivo,3810,669454.75,161.70
9,ferramentas_jardim,3448,567145.68,132.88


---

# Parte 3 — Window Functions

## Exercício 11 — Ranking das vendas por categoria

**Nível:** Fácil

Use `ROW_NUMBER()` para numerar as vendas dentro de cada categoria, da maior venda para a menor.

Retorne:
- `categoria`
- `order_id`
- `valor_total`
- `ranking_venda_categoria`

**Dica:** use `PARTITION BY p.product_category_name` e `ORDER BY f.valor_total DESC`.

In [ ]:
# Escreva sua consulta SQL aqui

query = """

"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 12 — Pedido mais caro de cada estado

**Nível:** Médio

Use `ROW_NUMBER()` para encontrar o pedido de maior valor em cada estado.

Retorne:
- `estado`
- `order_id`
- `valor_total`

**Dica:** crie uma CTE com o ranking e depois filtre `posicao = 1`.

In [ ]:
# Escreva sua consulta SQL aqui

query = """

"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 13 — Ranking de estados por receita

**Nível:** Médio

Calcule a receita total por estado e use `RANK()` para ranquear os estados com maior faturamento.

Retorne:
- `estado`
- `qtd_pedidos`
- `receita_total`
- `ranking_estado`

**Dica:** primeiro agregue por estado em uma CTE.

In [ ]:
# Escreva sua consulta SQL aqui

query = """

"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 14 — Comparando RANK e DENSE_RANK com empate

**Nível:** Médio

Crie uma CTE manual com os valores abaixo e compare `RANK()` e `DENSE_RANK()`:

| categoria | receita_total |
|---|---:|
| beleza_saude | 100000 |
| cama_mesa_banho | 90000 |
| esporte_lazer | 90000 |
| informatica_acessorios | 70000 |
| moveis_decoracao | 60000 |

Retorne:
- `categoria`
- `receita_total`
- `rank_categoria`
- `dense_rank_categoria`

**Dica:** este exercício força um empate para a diferença entre as funções aparecer.

In [ ]:
# Escreva sua consulta SQL aqui

query = """

"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 15 — Percentual da venda dentro da categoria

**Nível:** Médio/Difícil

Para cada venda, calcule quanto ela representa percentualmente dentro do total de sua categoria.

Retorne:
- `categoria`
- `order_id`
- `valor_total`
- `total_categoria`
- `percentual_na_categoria`

**Dica:** use `SUM(f.valor_total) OVER (PARTITION BY categoria)`.

In [ ]:
# Escreva sua consulta SQL aqui

query = """

"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 16 — Receita acumulada mensal por estado

**Nível:** Difícil

Calcule a receita mensal por estado e depois calcule a receita acumulada mês a mês.

Retorne:
- `estado`
- `ano`
- `mes`
- `ano_mes`
- `receita_mes`
- `receita_acumulada_estado`

**Dica:** primeiro crie uma CTE com a receita mensal por estado. Depois use `SUM() OVER()`.

In [ ]:
# Escreva sua consulta SQL aqui

query = """

"""

# Para executar:
# banco_dados.sql(query).df()

---

# Parte 4 — Índices e Otimização

## Exercício 17 — Índice para filtro por estado

**Nível:** Fácil

Considere a consulta:

```sql
SELECT
    cliente_sk,
    customer_id,
    customer_state
FROM dim_cliente_base
WHERE customer_state = 'SP';
```

Crie um índice que poderia ajudar essa consulta.

**Dica:** identifique a coluna usada no `WHERE` e crie o índice na tabela onde essa coluna existe.

In [ ]:
# Escreva sua consulta SQL aqui

query = """

"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 18 — Índice para JOIN e filtro por data

**Nível:** Médio

Considere a consulta:

```sql
SELECT
    f.venda_sk,
    f.order_id,
    f.valor_total,
    t.data
FROM fato_vendas_base f
INNER JOIN dim_tempo_base t
    ON f.tempo_sk = t.tempo_sk
WHERE t.data >= '2018-01-01';
```

Crie índices que poderiam ajudar essa consulta.

**Dicas:**
- `tempo_sk` é usado no `JOIN`.
- `data` é usada no filtro de período.
- `tempo_sk` não é uma data; é uma chave substituta.

In [ ]:
# Escreva sua consulta SQL aqui

query = """

"""

# Para executar:
# banco_dados.sql(query).df()

## Exercício 19 — Índice composto para filtro combinado

**Nível:** Difícil

Considere a consulta:

```sql
SELECT
    venda_sk,
    order_id,
    cliente_sk,
    produto_sk,
    valor_total
FROM fato_vendas_base
WHERE cliente_sk = 5000
  AND produto_sk = 15420;
```

Crie um índice composto adequado para essa consulta.

**Dica:** quando duas colunas aparecem juntas no `WHERE`, um índice composto pode ser útil.

In [ ]:
# Escreva sua consulta SQL aqui

query = """

"""

# Para executar:
# banco_dados.sql(query).df()

---

# Parte 5 — Aula 3 teórica: particionamento e materialized views

## Exercício 20 — Análise conceitual de performance em DW

**Nível:** Teórico

Responda com suas palavras.

Um dashboard de e-commerce mostra receita por mês, estado e categoria. Toda vez que ele abre, executa uma consulta com `JOIN` entre `fato_vendas`, `dim_tempo`, `dim_cliente` e `dim_produto`, além de `GROUP BY`.

Responda:

1. Por que essa consulta pode ficar lenta em uma base muito grande?
2. Qual seria a vantagem de particionar a tabela fato por uma coluna de data, como `data_venda`?
3. Qual seria a vantagem de criar uma Materialized View com receita por mês, estado e categoria?
4. Por que `tempo_sk` não deve ser tratado como data nesse dataset?

**Dica:** pense na diferença entre calcular tudo na hora e consultar um resumo já calculado.

### Espaço para resposta

Escreva sua resposta aqui.